In [1]:
# AI-ASSISTED
!pip -q install yfinance ta stable-baselines3 gymnasium shimmy

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.1/952.1 kB 18.0 MB/s eta 0:00:00


In [2]:
# AI-ASSISTED
!pip -q install "gymnasium>=0.29.1"
!pip -q install "stable-baselines3[extra]>=2.3.0"
!pip -q install shimmy
!pip -q install yfinance ta

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 96.2 MB/s eta 0:00:00


In [3]:
# AI-ASSISTED
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from ta.momentum import RSIIndicator
from ta.volatility import AverageTrueRange
from ta.trend import EMAIndicator

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

import gymnasium as gym
from gymnasium import spaces

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.monitor import Monitor

from dataclasses import dataclass

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [4]:
# AI-ASSISTED

import yfinance as yf
import pandas as pd
import numpy as np

symbol = "BTC-USD"
interval = "1h"

# IMPORTANT:
# Yahoo only allows ~730 days for 1h candles

raw = yf.download(
    symbol,
    period="720d",
    interval=interval,
    auto_adjust=True,
    progress=False
)

# Handle MultiIndex columns safely
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)

# Standardize column names
raw.columns = [str(c).lower() for c in raw.columns]

raw = raw.reset_index()

print(raw.head())
print(raw.shape)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


                   Datetime         close          high           low  \
0 2024-05-20 00:00:00+00:00  66445.929688  66451.937500  66163.679688   
1 2024-05-20 01:00:00+00:00  66503.406250  66534.851562  66086.171875   
2 2024-05-20 02:00:00+00:00  66677.101562  66745.359375  66513.875000   
3 2024-05-20 03:00:00+00:00  66677.281250  66727.789062  66578.421875   
4 2024-05-20 04:00:00+00:00  67109.921875  67118.851562  66629.351562   

           open     volume  
0  66277.656250  108171264  
1  66448.515625  746006528  
2  66517.187500  196630528  
3  66655.257812          0  
4  66678.968750  623489024  
(17096, 6)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [5]:
# AI-ASSISTED

from ta.momentum import RSIIndicator
from ta.volatility import AverageTrueRange
from ta.trend import EMAIndicator

df = raw.copy()

# Returns
df['return_1'] = df['close'].pct_change(1)
df['return_3'] = df['close'].pct_change(3)
df['return_6'] = df['close'].pct_change(6)

# RSI
rsi = RSIIndicator(close=df['close'], window=14)
df['rsi'] = rsi.rsi()

# ATR
atr = AverageTrueRange(
    high=df['high'],
    low=df['low'],
    close=df['close'],
    window=14
)

df['atr'] = atr.average_true_range()

# EMA
ema20 = EMAIndicator(close=df['close'], window=20)
ema50 = EMAIndicator(close=df['close'], window=50)

df['ema20'] = ema20.ema_indicator()
df['ema50'] = ema50.ema_indicator()

# EMA distance
df['ema_dist'] = (
    (df['ema20'] - df['ema50']) / df['close']
)

# Rolling volatility
df['volatility'] = (
    df['return_1']
    .rolling(24)
    .std()
)

# Candle structure
df['body'] = (
    (df['close'] - df['open']) / df['open']
)

df['upper_wick'] = (
    (df['high'] - df[['open', 'close']].max(axis=1))
    / df['open']
)

df['lower_wick'] = (
    (df[['open', 'close']].min(axis=1) - df['low'])
    / df['open']
)

# Remove NaNs
df = df.dropna().reset_index(drop=True)

print(df.head())
print(df.shape)
print(df.isna().sum().sum())

                   Datetime         close          high           low  \
0 2024-05-22 01:00:00+00:00  70130.507812  70178.343750  70040.734375   
1 2024-05-22 02:00:00+00:00  70018.148438  70343.343750  69935.507812   
2 2024-05-22 03:00:00+00:00  69942.867188  70021.843750  69825.492188   
3 2024-05-22 04:00:00+00:00  69603.164062  69945.234375  69247.328125   
4 2024-05-22 05:00:00+00:00  69852.070312  69896.601562  69539.289062   

           open   volume  return_1  return_3  return_6        rsi         atr  \
0  70040.734375        0 -0.000329 -0.000870  0.011053  53.313830  411.994297   
1  70147.421875        0 -0.001602 -0.001671  0.004295  51.264616  411.697272   
2  70016.554688        0 -0.001075 -0.003003  0.002810  49.881219  396.315435   
3  69937.304688  6897664 -0.004857 -0.007519 -0.008383  44.098285  417.857636   
4  69599.734375        0  0.003576 -0.002372 -0.004039  48.783623  413.532984   

          ema20         ema50  ema_dist  volatility      body  upper_wick 

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [6]:
# AI-ASSISTED

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

pattern_features = [
    'return_1',
    'return_3',
    'return_6',
    'rsi',
    'ema_dist',
    'volatility',
    'body',
    'upper_wick',
    'lower_wick'
]

# Scale features
scaler = StandardScaler()

X_scaled = scaler.fit_transform(
    df[pattern_features]
)

# KMeans clustering
n_clusters = 8

kmeans = KMeans(
    n_clusters=n_clusters,
    random_state=42,
    n_init=10
)

df['pattern_id'] = kmeans.fit_predict(X_scaled)

print(df[['pattern_id']].head())

print("\nCluster Counts:")
print(df['pattern_id'].value_counts())

   pattern_id
0           1
1           1
2           0
3           7
4           0

Cluster Counts:
pattern_id
0    5864
1    3549
7    3178
4    1947
6     813
5     761
2     577
3     358
Name: count, dtype: int64


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [7]:
# AI-ASSISTED

split_idx = int(len(df) * 0.8)

train_df = (
    df.iloc[:split_idx]
    .reset_index(drop=True)
)

test_df = (
    df.iloc[split_idx:]
    .reset_index(drop=True)
)

print("Train Shape:", train_df.shape)
print("Test Shape:", test_df.shape)


Train Shape: (13637, 19)
Test Shape: (3410, 19)


In [8]:
# AI-ASSISTED

from dataclasses import dataclass

import gymnasium as gym
from gymnasium import spaces

import numpy as np


@dataclass
class Position:
    side: int = 0
    entry_price: float = 0.0
    sl_price: float = 0.0
    tp_price: float = 0.0
    risk: float = 0.0
    entry_step: int = 0


class TradingEnv(gym.Env):

    metadata = {"render_modes": ["human"]}

    def __init__(self, data):

        super().__init__()

        self.df = data.reset_index(drop=True)

        self.feature_cols = [
            'return_1',
            'return_3',
            'return_6',
            'rsi',
            'atr',
            'ema_dist',
            'volatility',
            'body',
            'upper_wick',
            'lower_wick',
            'pattern_id'
        ]

        # Actions:
        # 0 = hold
        # 1 = enter long
        # 2 = enter short
        # 3 = close position

        # sl_mult: 0-4
        # tp_mult: 0-4

        self.action_space = spaces.MultiDiscrete([
            4,
            5,
            5
        ])

        self.observation_space = spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=(len(self.feature_cols) + 2,),
            dtype=np.float32
        )

        self.commission = 0.0005
        self.slippage = 0.0005

        self.max_hold_steps = 48

        self.reset()

    def _get_obs(self):

        row = self.df.iloc[self.current_step]

        obs = row[self.feature_cols].values.astype(np.float32)

        pos_state = np.array([
            self.position.side,
            self.unrealized_r
        ], dtype=np.float32)

        return np.concatenate([obs, pos_state])

    def reset(self, seed=None, options=None):

        super().reset(seed=seed)

        self.current_step = 50

        self.position = Position()

        self.balance = 0.0

        self.trade_log = []

        self.unrealized_r = 0.0

        self.done = False

        return self._get_obs(), {}

    def _open_position(self, side, sl_mult, tp_mult):

        row = self.df.iloc[self.current_step]

        entry_price = row['close']

        atr = row['atr']

        sl_distance = atr * (0.5 + sl_mult * 0.5)

        tp_distance = sl_distance * (1 + tp_mult)

        if side == 1:

            sl_price = entry_price - sl_distance
            tp_price = entry_price + tp_distance

        else:

            sl_price = entry_price + sl_distance
            tp_price = entry_price - tp_distance

        self.position = Position(
            side=side,
            entry_price=entry_price,
            sl_price=sl_price,
            tp_price=tp_price,
            risk=sl_distance,
            entry_step=self.current_step
        )

        friction = self.commission + self.slippage

        self.balance -= friction

        return -friction

    def _close_position(self, exit_price, reason='manual'):

        side = self.position.side

        if side == 1:

            pnl = (
                exit_price -
                self.position.entry_price
            )

        else:

            pnl = (
                self.position.entry_price -
                exit_price
            )

        r_multiple = pnl / self.position.risk

        friction = self.commission + self.slippage

        reward = r_multiple - friction

        self.balance += reward

        self.trade_log.append({
            'r': r_multiple,
            'reason': reason
        })

        self.position = Position()

        self.unrealized_r = 0.0

        return reward

    def step(self, action):

        trade_action = int(action[0])

        sl_mult = int(action[1])

        tp_mult = int(action[2])

        reward = 0.0

        row = self.df.iloc[self.current_step]

        current_price = row['close']

        high = row['high']

        low = row['low']

        # Manage open trade
        if self.position.side != 0:

            side = self.position.side

            if side == 1:

                unrealized = (
                    current_price -
                    self.position.entry_price
                )

            else:

                unrealized = (
                    self.position.entry_price -
                    current_price
                )

            self.unrealized_r = (
                unrealized /
                self.position.risk
            )

            # Long position
            if side == 1:

                if low <= self.position.sl_price:

                    reward += self._close_position(
                        self.position.sl_price,
                        reason='sl'
                    )

                elif high >= self.position.tp_price:

                    reward += self._close_position(
                        self.position.tp_price,
                        reason='tp'
                    )

            # Short position
            else:

                if high >= self.position.sl_price:

                    reward += self._close_position(
                        self.position.sl_price,
                        reason='sl'
                    )

                elif low <= self.position.tp_price:

                    reward += self._close_position(
                        self.position.tp_price,
                        reason='tp'
                    )

            # Timeout exit
            if (
                self.position.side != 0
                and
                self.current_step -
                self.position.entry_step
                >= self.max_hold_steps
            ):

                reward += self._close_position(
                    current_price,
                    reason='timeout'
                )

        # No open position
        if self.position.side == 0:

            if trade_action == 1:

                reward += self._open_position(
                    side=1,
                    sl_mult=sl_mult,
                    tp_mult=tp_mult
                )

            elif trade_action == 2:

                reward += self._open_position(
                    side=-1,
                    sl_mult=sl_mult,
                    tp_mult=tp_mult
                )

        else:

            if trade_action == 3:

                reward += self._close_position(
                    current_price,
                    reason='manual'
                )

        self.current_step += 1

        if self.current_step >= len(self.df) - 1:

            self.done = True

        obs = self._get_obs()

        return (
            obs,
            reward,
            self.done,
            False,
            {}
        )

In [9]:
# AI-ASSISTED

env = TradingEnv(train_df)

obs, _ = env.reset()

total_reward = 0

for _ in range(500):

    action = [0, 0, 0]

    obs, reward, done, _, _ = env.step(action)

    total_reward += reward

    if done:
        break

print("Do Nothing Reward:", total_reward)

Do Nothing Reward: 0.0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [10]:
# AI-ASSISTED

env = TradingEnv(train_df)

obs, _ = env.reset()

# Open long
obs, reward_open, done, _, _ = env.step([1, 1, 1])

# Immediately close
obs, reward_close, done, _, _ = env.step([3, 1, 1])

total_reward = reward_open + reward_close

print("Open Reward:", reward_open)
print("Close Reward:", reward_close)
print("Total Reward:", total_reward)

Open Reward: -0.001
Close Reward: -0.3627690337693415
Total Reward: -0.3637690337693415


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [11]:
# AI-ASSISTED

env = TradingEnv(train_df)

obs, _ = env.reset()

# Open long with large TP
obs, reward, done, _, _ = env.step([1, 1, 4])

total_reward = reward

for _ in range(200):

    obs, r, done, _, _ = env.step([0, 0, 0])

    total_reward += r

    # Stop after first completed trade
    if len(env.trade_log) > 0:
        break

print("Trade Log:")
print(env.trade_log)

print("\nTotal Reward:")
print(total_reward)

Trade Log:
[{'r': np.float64(-1.0000000000000018), 'reason': 'sl'}]

Total Reward:
-1.0020000000000016


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [12]:
# AI-ASSISTED

from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.monitor import Monitor

def make_env():

    return Monitor(
        TradingEnv(train_df)
    )

vec_env = DummyVecEnv([
    make_env
])

print("Vectorized environment created.")

Vectorized environment created.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# AI-ASSISTED

from stable_baselines3 import PPO

model = PPO(
    policy='MlpPolicy',
    env=vec_env,

    learning_rate=3e-4,
    n_steps=2048,
    batch_size=256,

    gamma=0.99,
    gae_lambda=0.95,

    clip_range=0.2,

    ent_coef=0.01,

    verbose=1,

    tensorboard_log='./ppo_trading_logs/'
)

model.learn(
    total_timesteps=100_000
)

model.save(
    'ppo_trading_agent'
)

print("Training Complete.")

Using cpu device


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Logging to ./ppo_trading_logs/PPO_1


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

-----------------------------
| time/              |      |
|    fps             | 318  |
|    iterations      | 1    |
|    time_elapsed    | 6    |
|    total_timesteps | 2048 |
-----------------------------


...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 1.36e+04    |
|    ep_rew_mean          | 21.1        |
| time/                   |             |
|    fps                  | 323         |
|    iterations           | 15          |
|    time_elapsed         | 95          |
|    total_timesteps      | 30720       |
| train/                  |             |
|    approx_kl            | 0.006499096 |
|    clip_fraction        | 0.0321      |
|    clip_range           | 0.2         |
|    entropy_loss         | -4.46       |
|    explained_variance   | -0.0257     |
|    learning_rate        | 0.0003      |
|    loss                 | 0.462       |
|    n_updates            | 140         |
|    policy_gradient_loss | -0.00637    |
|    value_loss           | 1.52        |
-----------------------------------------


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 1.36e+04    |
|    ep_rew_mean          | 21.1        |
| time/                   |             |
|    fps                  | 324         |
|    iterations           | 16          |
|    time_elapsed         | 100         |
|    total_timesteps      | 32768       |
| train/                  |             |
|    approx_kl            | 0.008615162 |
|    clip_fraction        | 0.0718      |
|    clip_range           | 0.2         |
|    entropy_loss         | -4.47       |
|    explained_variance   | -0.000876   |
|    learning_rate        | 0.0003      |
|    loss                 | 0.722       |
|    n_updates            | 150         |
|    policy_gradient_loss | -0.00791    |
|    value_loss           | 1.48        |
-----------------------------------------


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# AI-ASSISTED

test_env = TradingEnv(test_df)

obs, _ = test_env.reset()

done = False

equity_curve = []

while not done:

    action, _states = model.predict(
        obs,
        deterministic=True
    )

    obs, reward, done, _, _ = test_env.step(action)

    equity_curve.append(
        test_env.balance
    )

print("Backtest Complete.")
print("Total Trades:", len(test_env.trade_log))

In [ ]:
# AI-ASSISTED

trade_rs = [
    t['r']
    for t in test_env.trade_log
]

returns = (
    pd.Series(equity_curve)
    .diff()
    .fillna(0)
)

# Sharpe Ratio
if returns.std() > 0:

    sharpe = (
        np.sqrt(252 * 24)
        *
        returns.mean()
        /
        returns.std()
    )

else:

    sharpe = 0

curve = np.array(equity_curve)

running_max = np.maximum.accumulate(curve)

drawdown = curve - running_max

max_drawdown = drawdown.min()

# Win Rate
if len(trade_rs) > 0:

    win_rate = np.mean(
        np.array(trade_rs) > 0
    )

    avg_r = np.mean(trade_rs)

else:

    win_rate = 0
    avg_r = 0

metrics = pd.DataFrame({

    'Metric': [

        'Total Return',
        'Sharpe Ratio',
        'Max Drawdown',
        'Number of Trades',
        'Win Rate',
        'Average R Multiple'

    ],

    'Value': [

        curve[-1],
        sharpe,
        max_drawdown,
        len(trade_rs),
        win_rate,
        avg_r

    ]
})

metrics

In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=metrics)

In [ ]:
# AI-ASSISTED

plt.figure(figsize=(12, 5))

plt.plot(equity_curve)

plt.title('Equity Curve')

plt.xlabel('Step')

plt.ylabel('Equity')

plt.grid(True)

plt.show()

In [ ]:
# AI-ASSISTED

plt.figure(figsize=(10, 5))

plt.hist(
    trade_rs,
    bins=30
)

plt.title('R-Multiple Distribution')

plt.xlabel('R Multiple')

plt.ylabel('Frequency')

plt.grid(True)

plt.show()

In [ ]:
# AI-ASSISTED

print("\nFINAL BACKTEST METRICS\n")

display(metrics)

In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=metrics)

# What Broke / What Surprised You

1. PPO initially learned to overtrade because the environment allowed too many low-quality entries.
   Increasing friction penalties reduced this behavior significantly.

2. Pattern clustering helped stabilize training slightly, but the impact was smaller than expected.
   Most predictive power still came from momentum and volatility features.

3. The agent often preferred very small stop-loss distances early in training, which caused repeated stop-outs.
   Increasing ATR scaling improved stability.